## Configurar ambiente

In [ ]:
# instalar modulo en python
!pip install playwright
# instalar componentes en el server
!playwright install
# preparar el ambiente
!apt-get update
!apt-get install -y libnss3 libatk-bridge2.0-0 libgtk-3-0 libgbm-dev

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 9.5 MB/s eta 0:00:00
(node:26972) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 0.0s167.3 MiB [] 0% 41.4s167.3 MiB [] 0% 18.0s167.3 MiB [] 0% 15.4s167.3 MiB [] 0% 9.5s167.3 MiB [] 1% 5.6s167.3 MiB [] 2% 4.2s167.3 MiB [] 3% 3.5s167.3 MiB [] 3% 3.4s167.3 MiB [] 4% 3.5s167.3 MiB [] 5% 3.1s167.3 MiB [] 6% 2.8s167.3 MiB [] 7% 2.6s167.3 MiB [] 8% 2.5s167.3 MiB [] 8% 2.4s167.3 MiB [] 9% 2.4s167.3 MiB [] 10% 2.4s167.3 MiB [] 11% 2.4s167.3 MiB [] 12% 2.3s167.3 MiB [] 13% 2.2s167.3 MiB [] 13% 2.1s167.3 MiB [] 14% 2.1s167.3 MiB [] 15% 2.1s167.3 MiB [] 16% 2.0s167.3 MiB [] 17% 2.0s167.3 MiB [] 18% 1.9s167.3 MiB [] 19% 1.9s167.3 MiB [] 20% 1.8s167.3 MiB [] 21% 1.8s167.3 MiB [] 21% 1.7s167.

## Instalar modulos

In [ ]:
# importar los modulos
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from datetime import datetime
import os
import pandas as pd

## pull de data

### Definir variables

In [ ]:
# definir las variables
URL="https://www.metrocuadrado.com/apartamento-apartaestudio/arriendo/bogota/"

### Definir funciones
+ scroll_page : cargar toda la pagina
+ get_rendered_html : renderizar la pagina y pasarla a html
+ main : realiza la consulta de la url

In [ ]:
async def scroll_page(page):
    previous_count = 0
    no_change_count = 0

    for i in range(30):
        cards = await page.query_selector_all(".property-card__container")
        current_count = len(cards)

        print(f"Scroll {i}: {current_count}")

        if current_count == previous_count:
            no_change_count += 1
        else:
            no_change_count = 0

        if no_change_count >= 3:
            print("No cargan más resultados")
            break

        previous_count = current_count

        # scroll más natural
        await page.mouse.wheel(0, 2000)

        try:
            await page.wait_for_function(
                f"document.querySelectorAll('.property-card__container').length > {previous_count}",
                timeout=5000
            )
        except:
            pass

In [ ]:
async def get_rendered_html(page):
    # 🔥 Expandir Shadow DOM
    html = await page.evaluate("""
    () => {
        function expand(element) {
            if (element.shadowRoot) {
                element.innerHTML = element.shadowRoot.innerHTML;
            }
            element.querySelectorAll('*').forEach(el => {
                if (el.shadowRoot) {
                    el.innerHTML = el.shadowRoot.innerHTML;
                }
            });
        }

        expand(document.documentElement);
        return document.documentElement.outerHTML;
    }
    """)
    return html


In [ ]:
async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        page = await browser.new_page()

        await page.goto(URL)

        # esperar contenido inicial
        await page.wait_for_selector(".property-card__container")

        # 🔁 scroll para cargar todo
        await scroll_page(page)

        # 💾 obtener HTML renderizado con Shadow DOM expandido
        html = await get_rendered_html(page)

        # 📦 estructura tipo data lake
        fecha = datetime.now().strftime("%Y-%m-%d")
        path = f"data/raw/metrocuadrado/{fecha}"
        os.makedirs(path, exist_ok=True)

        file_path = f"{path}/pagina_1_rendered.html"

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(html)

        print(f"\n✅ HTML renderizado guardado en:\n{file_path}")

        await browser.close()

In [ ]:
await main()

Scroll 0: 6
Scroll 1: 12
Scroll 2: 18
Scroll 3: 68
Scroll 4: 68
Scroll 5: 68
Scroll 6: 68
No cargan más resultados

✅ HTML renderizado guardado en:
data/raw/metrocuadrado/2026-03-31/pagina_1_rendered.html


## limpieza de la data

### Validaciones inciales

In [ ]:
fecha = datetime.now().strftime("%Y-%m-%d")
path = f"data/raw/metrocuadrado/{fecha}"

file_path = f"{path}/pagina_1_rendered.html"

# lee los datos y valida que esten
with open(file_path, "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")
cards = soup.find_all("div", class_="property-card__container")

print("Total de cards:", len(cards))

Total de cards: 68


In [ ]:
# imprimir el elemento 1
cards[0]

<div class="property-card__container property-card__highlighted"><div class="property-card__content"><a href="/inmueble/arriendo-apartamento-bogota-bellavista-1-habitaciones-2-banos-1-garajes/3072-M6017977?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F" rel="noreferrer" target="_blank"><div class="property-card__photo"><img alt="Foto de Apartamento en Arriendo en ROSALES BELLAVISTA, Bogotá D.C. con 1 habitaciones, 2 baños, área 90 m2, 1 garaje - 3072-M6017977" data-nimg="1" decoding="async" height="217.33" src="https://multimedia.metrocuadrado.com/3072-M6017977/3072-M6017977_17_p.jpg" style="color: transparent;" title="Apartamento en Arriendo en ROSALES BELLAVISTA, Bogotá D.C. - 3072-M6017977" width="324"/><div class="property-card__photo-tags"><pt-tag big-type="clientes" class="hydrated" element-id="higlightTag" icon="crown" size="small" type="blue"><div id="higlightTag"><div class="pt-tag pt-tag--blue pt-tag--small"><pt-text class="hydrated"><slot></slot></pt-text><pt-ic

In [ ]:
#href = cards[0].find_all("div", class_="property-card__content")[0].find_all("a")[0]["href"]
href = cards[0].find_all("a")[0]["href"]
href

'/inmueble/arriendo-apartamento-bogota-bellavista-1-habitaciones-2-banos-1-garajes/3072-M6017977?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F'

In [ ]:
url_img = cards[0].find_all("img")[0]["src"]
url_img

'https://multimedia.metrocuadrado.com/3072-M6017977/3072-M6017977_17_p.jpg'

In [ ]:
alt_img = cards[0].find_all("img")[0]["alt"]
alt_img

'Foto de Apartamento en Arriendo en ROSALES BELLAVISTA, Bogotá D.C. con 1 habitaciones, 2 baños, área 90 m2, 1 garaje - 3072-M6017977'

In [ ]:
barrio = cards[0].find_all("div", class_="property-card__detail-top__left")[0].find_all("div")[0].get_text(strip=True).split("|")[0]
barrio

'ROSALES BELLAVISTA '

In [ ]:
titulo = cards[0].find_all("div", class_="property-card__detail-title")[0].find_all("h2")[0].get_text(strip=True)
titulo

'Apartamento en Arriendo, ROSALES BELLAVISTA, Bogotá D.C.'

In [ ]:
precio_str = cards[0].find_all("div", class_="property-card__detail-price")[0].get_text(strip=True)
precio_str

'$9.000.000'

In [ ]:
atributos = cards[0].find_all("div", class_="pt-main-specs--feature")
list_atributos = [ a.get_text(strip=True)  for a in atributos ]

In [ ]:
def filtrar_atributos(lista_atributos):

  # se define un esquema inicial
  esquema_salida = {
      "area" : -1,
      "habitaciones" : -1,
      "banos" : -1,
      "parqueaderos" : 0
  }

  # aplciar filtros

  for item in lista_atributos:
    if 'm²' in item:
        esquema_salida['area'] = float(item.split()[0].replace(",","."))
    elif 'hab' in item:
        esquema_salida['habitaciones'] = int(item.split()[0])
    elif 'bañ' in item:
        esquema_salida['banos'] = float(item.split()[0])
    elif 'par' in item:
        esquema_salida['parqueaderos'] = int(item.split()[0])

  return esquema_salida

In [ ]:
resultado_filtro = filtrar_atributos(list_atributos)

In [ ]:
area = resultado_filtro["area"]
area

90.0

In [ ]:
hab = resultado_filtro["habitaciones"]
hab

1

In [ ]:
bano = resultado_filtro["banos"]
bano

2.0

In [ ]:
parq_cant = resultado_filtro["parqueaderos"]
parq_cant

1

### Limpieza

In [ ]:
precio = float(precio_str.replace("$","").replace(".",""))
precio

9000000.0

### Crear funciones

In [ ]:
def get_href(card):
  """
  retorna el href del inmueble tomado
  """
  href = card.find_all("a")[0]["href"]

  return href

def get_url_img(card):
  """
  retorna la url de la imagen
  """
  url_img = card.find_all("img")[0]["src"]

  return url_img

def get_alt_img(card):
  """
  retorna la informacion de alt de la imagen
  """

  alt_img = card.find_all("img")[0]["alt"]

  return alt_img


def get_barrio(card):
  """
  retorna la informacion del barrio
  """

  barrio = card.find_all("div", class_="property-card__detail-top__left")[0].find_all("div")[0].get_text(strip=True).split("|")[0]

  return barrio


def get_titulo(card):
  """
  retorna el titulo
  """
  titulo = card.find_all("div", class_="property-card__detail-title")[0].find_all("h2")[0].get_text(strip=True)

  return titulo

def get_precio_str(card):
  """
  retorna el precio en str
  """
  precio_str = card.find_all("div", class_="property-card__detail-price")[0].get_text(strip=True)

  return precio_str

def get_precio(precio_str):
  """
  retorna el precio en numeros
  """
  precio = int(precio_str.replace("$","").replace(".",""))

  return precio
def get_atributos(card):
  """
  Funcion que retorna los atributos de area,hab., banos. paq.
  """
  atributos = card.find_all("div", class_="pt-main-specs--feature")
  list_atributos = [ a.get_text(strip=True)  for a in atributos ]

  return list_atributos

def get_area(resultado_filtro):
  """
  retorna el area del inmueble
  """
  area = resultado_filtro["area"]

  return area

def get_habitaciones(resultado_filtro):
  """
  retorna la cantidad de habitaciones
  """
  hab = resultado_filtro["habitaciones"]

  return hab

def get_bano(resultado_filtro):
  """
  retorna la cantidad de baños, existe la posibilida de que sea 0.5 bano
  """
  bano = resultado_filtro["banos"]

  return bano

def get_parq_cant(resultado_filtro):
  """
  retorna la cantidad de parqueaderos
  """
  parq_cant = resultado_filtro["parqueaderos"]
  return parq_cant

In [ ]:
def get_esquema(card):
  """
  """
  href = get_href(card)
  url_img = get_url_img(card)
  alt_img = get_alt_img(card)
  barrio = get_barrio(card)
  titulo = get_titulo(card)
  precio_str = get_precio_str(card)
  lista_atributos = get_atributos(card)
  resultado_filtro = filtrar_atributos(lista_atributos)
  area = get_area(resultado_filtro)
  hab = get_habitaciones(resultado_filtro)
  bano = get_bano(resultado_filtro)
  parq_cant = get_parq_cant(resultado_filtro)

  # limpieza

  precio = get_precio(precio_str)




  return [href, url_img, alt_img, barrio, titulo, precio_str, precio, area, hab, bano, parq_cant]

In [ ]:
datos = []
for card in cards:

  formato = get_esquema(card)

  print(formato)
  datos.append(formato)

['/inmueble/arriendo-apartamento-bogota-bellavista-1-habitaciones-2-banos-1-garajes/3072-M6017977?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F', 'https://multimedia.metrocuadrado.com/3072-M6017977/3072-M6017977_17_p.jpg', 'Foto de Apartamento en Arriendo en ROSALES BELLAVISTA, Bogotá D.C. con 1 habitaciones, 2 baños, área 90 m2, 1 garaje - 3072-M6017977', 'ROSALES BELLAVISTA ', 'Apartamento en Arriendo, ROSALES BELLAVISTA, Bogotá D.C.', '$9.000.000', 9000000, 90.0, 1, 2.0, 1]
['/inmueble/arriendo-apartamento-bogota-chico-norte-et-ii-1-habitaciones-2-banos-2-garajes/754-M6524317?src_url=%2Fapartamento-apartaestudio%2Farriendo%2Fbogota%2F', 'https://multimedia.metrocuadrado.com/754-M6524317/754-M6524317_5_p.jpg', 'Foto de Apartamento en Arriendo en CHICO RESERVADO, Bogotá D.C. con 1 habitaciones, 2 baños, área 99 m2, 2 garaje - 754-M6524317', 'CHICO RESERVADO ', 'Apartamento en Arriendo, CHICO RESERVADO, Bogotá D.C.', '$6.400.000', 6400000, 99.0, 1, 2.0, 2]
['/inmueble/arr

68

In [ ]:
df = pd.DataFrame(datos)

In [ ]:
df.to_csv("salida.csv")

In [ ]:
df

,0,1,2,3,4,5,6,7,8,9,10
0,/inmueble/arriendo-apartamento-bogota-bellavis...,https://multimedia.metrocuadrado.com/3072-M601...,Foto de Apartamento en Arriendo en ROSALES BEL...,ROSALES BELLAVISTA,"Apartamento en Arriendo, ROSALES BELLAVISTA, B...",$9.000.000,9000000,90.00,1,2.0,1
1,/inmueble/arriendo-apartamento-bogota-chico-no...,https://multimedia.metrocuadrado.com/754-M6524...,Foto de Apartamento en Arriendo en CHICO RESER...,CHICO RESERVADO,"Apartamento en Arriendo, CHICO RESERVADO, Bogo...",$6.400.000,6400000,99.00,1,2.0,2
2,/inmueble/arriendo-apartamento-bogota-chapiner...,https://multimedia.metrocuadrado.com/4186-M635...,Foto de Apartamento en Arriendo en Chapinero A...,Chapinero Alto,"Apartamento en Arriendo, Chapinero Alto, Bogot...",$3.100.000,3100000,50.00,1,1.0,1
3,/inmueble/arriendo-apartamento-bogota-san-migu...,https://multimedia.metrocuadrado.com/20449-M65...,"Foto de Apartamento en Arriendo en San Miguel,...",San Miguel,"Apartamento en Arriendo, San Miguel, Bogotá D.C.",$2.200.000,2200000,45.00,1,2.0,1
4,/inmueble/arriendo-apartamento-bogota-cjr-neos...,https://multimedia.metrocuadrado.com/MC5729420...,Foto de Apartamento en Arriendo en LOS ROSALES...,LOS ROSALES CHICO Chicó,"Apartamento en Arriendo, LOS ROSALES CHICO C...",$3.450.000,3450000,39.00,1,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...
63,/inmueble/arriendo-apartaestudio-bogota-san-vi...,https://multimedia.metrocuadrado.com/2361-M619...,Foto de Apartaestudio en Arriendo en San Victo...,San Victorino,"Apartaestudio en Arriendo, San Victorino, Bogo...",$2.200.000,2200000,47.45,1,2.0,0
64,/inmueble/arriendo-apartamento-bogota-chico-no...,https://multimedia.metrocuadrado.com/MC6500818...,Foto de Apartamento en Arriendo en CHICO NORTE...,CHICO NORTE Zona Urbana,"Apartamento en Arriendo, CHICO NORTE Zona Ur...",$4.600.000,4600000,63.00,1,2.0,1
65,/inmueble/arriendo-apartamento-bogota-el-plan-...,https://multimedia.metrocuadrado.com/MC6411937...,Foto de Apartamento en Arriendo en COLINA EL P...,COLINA EL PLAN Colina y Alrededores,"Apartamento en Arriendo, COLINA EL PLAN Coli...",$2.000.000,2000000,36.00,1,1.0,0
66,/inmueble/arriendo-apartamento-bogota-mochuelo...,https://multimedia.metrocuadrado.com/MC6441923...,Foto de Apartamento en Arriendo en MOCHUELO NO...,MOCHUELO NORTE Santa Bárbara,"Apartamento en Arriendo, MOCHUELO NORTE Sant...",$2.800.000,2800000,25.00,1,1.0,0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO
+ poner nombre a las columnas
+ realizar descriptiva de los datos
  + valiar los tipos de variables y que metricas se debe usuara para su descriptiva
+ normalizacion
+ realizar graficas para las columnas
+ crear una columna para el id que se toma del href (Ejemplo el primero es 3072-M6017977)

### Descriptiva

1. Cuales son los barrios de las ofertas y cual es el que mas se repite
1. Cual es el barrio con un promedio de arriendo alto, bajo y el medio
1. En terminos de area cuadrada cual es el barrio mas caro, bajo y medio
1. que influye mas en el costo, el area, el barrio o la cantidad de hab,baños o paqueaderos.